# 00 · 환경 점검 & GPU 컴퓨팅 / CuPy 개론

> **CuPy 2일 집중 코스 — Day 1 / 단원 1 (GPU 컴퓨팅과 CuPy 개론, 1.5H)**

이 노트북은 코스의 출발점입니다. GPU가 *왜* 빠른지, CuPy가 *무엇*인지 개념을 잡고,
실습 환경을 점검한 뒤 첫 GPU 연산을 실행합니다.

## 학습 목표
- CPU와 GPU의 구조적 차이(지연시간 vs 처리량)를 설명할 수 있다.
- GPU 프로그래밍 모델(host/device, 커널, grid·block·thread, SIMT)의 큰 그림을 안다.
- CuPy가 NumPy/SciPy의 드롭인 대체임을 이해하고 첫 연산을 실행한다.
- **GPU 연산은 비동기**라는 사실과, 올바른 시간 측정·전송 비용을 체감한다.

## 목차 (Table of Contents)
1. [CPU vs GPU — 왜 GPU인가](#1)
2. [GPU 프로그래밍 모델](#2)
3. [CuPy란 무엇인가](#3)
   - [3.1 CuPy ≠ NumPy 차이점](#3-1)
   - [3.2 CuPy 루틴 지도](#3-2)
4. [실습 환경 설정 & 점검](#4)
5. [첫 GPU 연산](#5)
6. [비동기 타이밍의 함정](#6)
   - [6.1 일회성 오버헤드와 워밍업](#6)
7. [전송 비용 (host ↔ device)](#7)
8. [연습문제](#8)
9. [체크포인트](#9)

### 실습 규칙 (중요)
- GPU 연산은 기본적으로 **비동기(async)** 입니다. 시간 측정은 **동기화(synchronize) 후** 또는 `course_utils.bench`(CUDA 이벤트 기반)로 하세요.
- `cp.asnumpy()` / `np.array(cupy배열)` 호출은 **device→host 전송**입니다. 결과 출력/플롯 등 **마지막 단계에서만** 최소화하세요.
- 정확성 검증은 `numpy.testing.assert_allclose`를 사용합니다. (float32 GPU 결과는 허용오차 `rtol/atol`을 약간 키웁니다.)

<a id="1"></a>
## 1. CPU vs GPU — 왜 GPU인가

CPU와 GPU는 **설계 철학**이 다릅니다.

| 구분 | CPU | GPU |
|------|-----|-----|
| 코어 | 소수(수~수십 개)의 **강력한** 코어 | 수천 개(1,000~10,000+)의 **단순한** 코어 |
| 최적화 목표 | **지연시간(latency)** 최소화 — 한 작업을 빨리 | **처리량(throughput)** 최대화 — 많은 작업을 동시에 |
| 메모리 대역폭 | 대략 ~100 GB/s | 대략 500 GB/s ~ 2 TB/s |
| 강점 | 분기 많은 순차 로직, OS/DB, 제어 흐름 | 대규모 **데이터 병렬** 연산(행렬, 신호, 이미지) |

GPU는 개별 코어의 클럭은 낮지만, **같은 연산을 거대한 데이터에 동시에** 적용하는 작업에서 압도적입니다.
NumPy의 `a * b`처럼 원소별·벡터화된 연산이 바로 그런 형태라 GPU 가속의 대상이 됩니다.

> **핵심 직관**: GPU는 '한 명의 천재'가 아니라 '수천 명의 일꾼'입니다. 일을 잘게 나눠 동시에 줄 수 있을 때 빛납니다.
> 반대로 작은 배열이나 순차 의존이 강한 작업은 GPU가 오히려 느릴 수 있습니다(런치 오버헤드 + 전송 비용).

<a id="2"></a>
## 2. GPU 프로그래밍 모델

**Host(CPU)와 Device(GPU)는 서로 다른 메모리 공간**을 가집니다. 큰 그림은 다음과 같습니다.
1. host 메모리(NumPy)에서 device 메모리(CuPy 배열)로 데이터를 **전송**한다.
2. device에서 **커널(kernel)** — GPU에서 실행되는 함수 — 을 실행해 연산한다.
3. 필요한 결과만 다시 host로 **전송**해 가져온다.

커널은 수많은 **스레드(thread)** 로 실행됩니다. 스레드는 다음 계층으로 조직됩니다.
- **thread** → 가장 작은 실행 단위(보통 원소 1개 담당)
- **block** → 스레드들의 묶음(공유 메모리를 공유)
- **grid** → 블록들의 묶음(전체 문제 공간)

하드웨어는 스레드를 **warp(32개)** 단위로 묶어 **SIMT**(Single Instruction, Multiple Threads) — *한 명령을 여러 스레드가 동시에* — 방식으로 실행합니다.

> 좋은 소식: CuPy로 NumPy 코드를 포팅할 때는 이 grid/block/thread를 **직접 다루지 않아도** 됩니다.
> CuPy가 알아서 커널을 생성·실행합니다. 직접 제어는 Day 2의 커널 작성(단원 5~6)에서 배웁니다.

호스트(CPU)에서는 **모든 파이썬**이 돌지만, 디바이스(GPU)에서 실행되는 커널은 **파이썬의 일부(subset)** 만 사용할 수 있습니다.

<img src="images/figures/new_host_device_code.png" width="600">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

<a id="3"></a>
## 3. CuPy란 무엇인가

**CuPy는 GPU에서 동작하는 NumPy/SciPy 라이브러리**로, 기존 코드를 NVIDIA CUDA(또는 AMD ROCm)에서 실행하는 **드롭인 대체(drop-in replacement)** 를 지향합니다.

- NumPy/SciPy와 **약 80~90% 호환** — `import numpy as np`를 `import cupy as cp`로 바꾸면 대부분 그대로 동작합니다.
- **메모리 할당과 GPU 커널 실행을 CuPy가 대신** 처리합니다(사용자가 CUDA를 직접 쓰지 않아도 됨).
- 오픈소스이며 **NVIDIA CUDA & AMD ROCm** 백엔드를 지원합니다.
- 내부적으로 NVIDIA 검증 라이브러리로 구동: **cuBLAS**(선형대수), **cuFFT**(FFT), **cuSPARSE**(희소행렬), **cuSOLVER**, **cuRAND**(난수), Thrust/CUB 등.
- **상호운용 표준** 지원: DLPack, CUDA Array Interface, `__array_function__`(NEP 18) 등 → PyTorch·TensorFlow 등과 무복사 연동(단원 7).
- 필요하면 **사용자 정의 CUDA 커널**(Elementwise/Reduction/RawKernel)도 작성 가능(Day 2).

> NVIDIA GTC 자료 기준, **거의 동일한 코드로 약 10배 빠른** 사례가 보고됩니다(예: Quadro RTX 8000). 다만 가속 폭은 문제 크기·하드웨어에 따라 달라집니다.

```python
# NumPy
import numpy as np
x = np.arange(1_000_000)
y = np.sin(x).sum()

# CuPy — 거의 동일!
import cupy as cp
x = cp.arange(1_000_000)
y = cp.sin(x).sum()   # GPU에서 실행
```

<a id="3-1"></a>
### 3.1 CuPy ≠ NumPy: 알아둘 차이점

"거의 같다"지만 **똑같지는 않습니다.** 다음 차이는 실습 중 버그로 이어지기 쉬우니 미리 알아둡니다.
(출처: NVIDIA GTC 강의자료 + CuPy 공식 문서 *Differences between CuPy and NumPy* — 두 출처에서 교차확인)

| 항목 | NumPy | CuPy |
|------|-------|------|
| 함수 커버리지 | 전체 | **대부분 지원**(일부 함수 없음) |
| dtype | 문자열·object·구조체 가능 | **숫자형 위주**(문자열/object 미지원, 구조체 매우 제한적) |
| 경계 밖 정수 인덱싱 | **에러(IndexError)** | **wrap-around** (조용히 처리 → 버그 주의!) |
| 메모리 전송 | 불필요 | 다른 라이브러리(OpenCV·matplotlib 등)와 주고받으려면 **명시적 전송 필요** |
| 타입 승격/캐스팅 | 안전하지만 느릴 수 있음 | **속도 우선** — 일부 캐스팅 결과가 다름(예: 음수 float→uint) |
| 난수 | — | 알고리즘이 달라 **bit 단위로 동일하지 않음**(cuRAND) |
| 리덕션 결과 | 스칼라(`np.float32`) | **0-차원 `cupy.ndarray`** (불필요한 동기화 회피 위함; 스칼라가 필요하면 `float()`/`.item()`) |
| ufunc 입력 | list·np.ndarray도 허용 | **CuPy 배열/스칼라만** 허용 |

> ⚠️ **직렬 for 루프를 피하세요.** 파이썬 `for`로 원소를 하나씩 처리하면 GPU에서는 CPU보다 **수십~수백 배 느려질 수 있습니다.**
> 항상 **벡터화된 배열 연산**(`a * b + c`, `cp.sum`, 슬라이싱)으로 표현하세요. 이것이 GPU 가속의 핵심 전제입니다.

<a id="3-2"></a>
### 3.2 CuPy 루틴 지도 (공식 overview)

CuPy 공식 [overview](https://docs.cupy.dev/en/stable/overview.html)는 라이브러리를 **4대 구성**으로 소개합니다. 각 루틴은 NVIDIA CUDA 라이브러리가 구동합니다.

| 구성 | 모듈 (공식 레퍼런스) | 백엔드 |
|------|----------------------|--------|
| **N차원 배열** | [`cupy.ndarray`](https://docs.cupy.dev/en/stable/reference/ndarray.html) | — |
| **NumPy 루틴** | [`cupy.*`](https://docs.cupy.dev/en/stable/reference/routines.html) · [`linalg`](https://docs.cupy.dev/en/stable/reference/linalg.html) · [`fft`](https://docs.cupy.dev/en/stable/reference/fft.html) · [`random`](https://docs.cupy.dev/en/stable/reference/random.html) | cuBLAS·cuSOLVER·cuFFT·cuRAND |
| **SciPy 루틴** | [`fft`](https://docs.cupy.dev/en/stable/reference/scipy_fft.html) · [`linalg`](https://docs.cupy.dev/en/stable/reference/scipy_linalg.html) · [`ndimage`](https://docs.cupy.dev/en/stable/reference/scipy_ndimage.html) · [`special`](https://docs.cupy.dev/en/stable/reference/scipy_special.html) · [`signal`](https://docs.cupy.dev/en/stable/reference/scipy_signal.html) · [`stats`](https://docs.cupy.dev/en/stable/reference/scipy_stats.html) | cuFFT·cuSOLVER 등 |
| **희소행렬** | [`cupyx.scipy.sparse`](https://docs.cupy.dev/en/stable/reference/scipy_sparse.html) · [`sparse.linalg`](https://docs.cupy.dev/en/stable/reference/scipy_sparse_linalg.html) | cuSPARSE |

또한 직접 [커스텀 CUDA 커널](https://docs.cupy.dev/en/stable/user_guide/kernel.html)(Elementwise/Reduction/Raw/JIT/Fusion)을 작성할 수 있고(→ Day 2),
DLPack·CUDA Array Interface 등 표준으로 PyTorch·TensorFlow와 [상호운용](https://docs.cupy.dev/en/stable/user_guide/interoperability.html)됩니다(→ 단원 7).

> **코스 매핑**: ndarray→`02`, NumPy 루틴→`03`, SciPy 루틴→`04`, 메모리/스트림→`05`/`06`, 커널→Day 2.

<a id="4"></a>
## 4. 실습 환경 설정 & 점검

CuPy는 CUDA 버전에 맞는 휠을 설치합니다(택1).
```bash
pip install cupy-cuda12x      # CUDA 12.x
# 또는
pip install cupy-cuda11x      # CUDA 11.x
pip install numpy scipy matplotlib
```
쉘명령

In [ ]:
! pip install cupy-cuda12x 

In [ ]:
! pip install numpy scipy matplotlib numba

아래 셀로 환경을 점검합니다. GPU 이름과 VRAM이 보이면 준비 완료입니다.

In [ ]:
# 공통 셋업: 패키지 + 코스 유틸리티 로드
import os, sys, time, math
import numpy as np

try:
    import cupy as cp
except Exception as e:
    raise RuntimeError(
        "CuPy import 실패. cupy-cuda12x 또는 cupy-cuda11x를 설치하세요.\n"
        f"원인: {e}"
    ) from e

from course_utils import print_env, bytes_human, bench, gpu_ms, cpu_ms

print_env()

### 📦 `course_utils` 함수 — 이번 노트북에서 도입

실습 공통 유틸리티는 `course_utils.py`에 모아두고 노트북이 진행되며 **필요한 함수만 하나씩** 추가합니다. 00에서 도입하는 함수:

- **`bytes_human(n)`** — 바이트 수를 `KB/MB/GB` 문자열로 변환(메모리 크기 출력용).
- **`print_env()`** — numpy·cupy 버전, GPU 이름, VRAM을 한 번에 출력(바로 위 셀에서 실행).
- **`bench(fn, n_repeat=20, n_warmup=3)`** — `cupyx.profiler.benchmark` 래퍼. 워밍업·동기화·반복평균을 자동 처리해 **비동기 GPU를 올바르게 측정**합니다.
- **`gpu_ms(r)` / `cpu_ms(r)`** — `bench` 결과에서 GPU 커널 / CPU wall-clock 평균 시간(ms)을 추출.

<a id="5"></a>
## 5. 첫 GPU 연산

`cp.arange`, `cp.sin` 등은 NumPy와 동일하게 쓰되 **결과가 GPU 메모리에 만들어집니다**.
마지막에 `cp.asnumpy`로 host로 가져와 확인합니다.

In [ ]:
x = cp.arange(10, dtype=cp.float32)   # GPU에 배열 생성
y = cp.sin(x) + 2.0                    # GPU에서 계산 (커널 자동 생성/실행)

print('타입      :', type(x))          # cupy.ndarray
print('디바이스  :', x.device)         # 어느 GPU에 있는지
print('GPU 결과  :', y[:5])
print('host 전송 :', cp.asnumpy(y)[:5])  # device -> host 복사

<a id="6"></a>
## 6. 비동기 타이밍의 함정

GPU 연산은 **비동기**입니다. 파이썬은 커널을 GPU에 '제출'만 하고 즉시 다음 줄로 넘어가므로,
`time.perf_counter()`로 감싸면 **연산이 끝나기 전 시간**이 찍혀 실제보다 빠르게 보입니다.
올바른 측정은 (a) `synchronize()`로 완료를 기다리거나, (b) `bench`(CUDA 이벤트 기반)를 쓰는 것입니다.

In [ ]:
n = 100_000_000 # n을 변경해보세요.

def work():
    # static 할당
    # a = cp.ones(n, dtype=cp.float32)
    # random number generation
    a = cp.random.random(n, dtype=cp.float32)
    return (a * 1.0001 + 2.0).sum()

# (1) 동기화 없음 — 잘못된 측정 (커널 제출 시간만 잼)
t0 = time.perf_counter(); _ = work(); t1 = time.perf_counter()
print(f'[잘못] sync 없음 : {(t1 - t0) * 1e3:8.3f} ms')

# (2) 직접 동기화 — 올바른 측정
t0 = time.perf_counter(); _ = work(); cp.cuda.Device().synchronize(); t1 = time.perf_counter()
print(f'[정상] sync 포함 : {(t1 - t0) * 1e3:8.3f} ms')

# (3) 권장 — cupyx.profiler.benchmark 래퍼 (워밍업 + 반복 평균)
r = bench(work, n_repeat=20, n_warmup=3)
print(f'[권장] bench GPU : {gpu_ms(r):8.3f} ms (CPU 런치 {cpu_ms(r):.3f} ms)')
print(r)

### 6.1 일회성 오버헤드와 워밍업

첫 측정이 유난히 느린 데는 이유가 있습니다(공식 문서가 "One-Time Overheads"로 안내).
- **컨텍스트 초기화**: 프로세스에서 **첫 CUDA 호출** 시 드라이버가 CUDA 컨텍스트를 만드느라 **수 초**가 걸릴 수 있습니다.
- **커널 컴파일(JIT)**: CuPy는 인자의 shape/dtype에 맞춰 커널을 **즉석 컴파일**합니다. 결과는 프로세스 내 캐시 + 디스크(`~/.cupy/kernel_cache`, 환경변수 `CUPY_CACHE_DIR`)에 저장되어 다음 호출부터 빨라집니다.

그래서 측정에는 **워밍업**이 필수입니다(`bench`가 자동 처리). 주피터에서는 전용 매직 `%gpu_timeit`도 편리합니다:

```python
%load_ext cupyx.profiler
%gpu_timeit (cp.random.random(1_000_000, dtype=cp.float32) ** 2).sum()
```

<a id="7"></a>
## 7. 전송 비용 (host ↔ device)

`cp.asnumpy()`(device→host)와 `cp.asarray()`(host→device)는 **PCIe를 통한 복사**라 비용이 큽니다.
연산은 GPU에 **머무르게** 하고 전송은 꼭 필요한 순간에만 하세요. 아래는 같은 연산을 '전송 없음 vs 포함'으로 비교합니다.

In [ ]:
n = 20_000_000  # n을 변경해보세요.
a = cp.random.random(n, dtype=cp.float32)

# 연산만: 결과 스칼라도 GPU에 둠
r_compute = bench(lambda: (a * 2.0 + 1.0).sum(), n_repeat=20)
# 연산 + host 전송: 매번 asnumpy 호출
r_e2e     = bench(lambda: cp.asnumpy((a * 2.0 + 1.0).sum()), n_repeat=20)

print(f'연산만          : {gpu_ms(r_compute):8.3f} ms')
print(f'연산+전송(e2e)  : {gpu_ms(r_e2e):8.3f} ms')
print('=> 반복문 안에서 asnumpy를 호출하면 전송이 병목이 됩니다. (단원 3에서 더 깊이 다룸)')

<a id="8"></a>
## 8. 연습문제 — NumPy → CuPy 포팅

아래 `feature_cpu`는 입력을 **z-score 표준화**한 뒤 **[0, 1]로 클리핑**합니다.
`feature_gpu`를 CuPy로 완성하세요. 조건:
- CuPy 연산만 사용하고, **마지막까지 GPU에 머무르게** 합니다(중간 `asnumpy` 금지).
- 검증 셀의 주석을 해제해 `assert_allclose`가 통과하는지 확인합니다.

In [ ]:
def feature_cpu(x_np):
    mu = x_np.mean(); sd = x_np.std() + 1e-8
    z = (x_np - mu) / sd
    return np.clip(z, 0.0, 1.0)

def feature_gpu(x_cp):
    # TODO: 위와 동일한 결과를 CuPy로 구현하세요 (cp.* 함수만 사용)
    raise NotImplementedError

# 검증 (구현 후 아래 두 줄 주석 해제)
x_np = np.random.randn(1_000_000).astype(np.float32)
ref = feature_cpu(x_np)
# out = cp.asnumpy(feature_gpu(cp.asarray(x_np)))
# np.testing.assert_allclose(ref, out, rtol=1e-5, atol=1e-5); print('OK')

<details>
<summary>💡 해답 보기</summary>

```python
def feature_gpu(x_cp):
    mu = x_cp.mean(); sd = x_cp.std() + 1e-8
    z = (x_cp - mu) / sd
    return cp.clip(z, 0.0, 1.0)

x_np = np.random.randn(1_000_000).astype(np.float32)
ref = feature_cpu(x_np)
out = cp.asnumpy(feature_gpu(cp.asarray(x_np)))
np.testing.assert_allclose(ref, out, rtol=1e-5, atol=1e-5)
print('OK')
```

포인트: `np.` → `cp.` 치환만으로 동작합니다. `mean/std/clip`이 모두 GPU에서 실행되고,
`asnumpy`는 **검증을 위한 마지막 한 번**만 호출합니다.
</details>

## 🧪 추가 실험 — 전송 비용과 대역폭

`asnumpy`(device→host) 전송 시간을 크기별로 재고 **유효 대역폭(GB/s)** 을 추정해 보세요.
> 예측 먼저: 크기가 100배 늘면 전송 시간도 100배일까요? 대역폭은 일정할까요?

In [ ]:
for n in [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]:
    a = cp.random.random(n, dtype=cp.float32)
    r = bench(lambda a=a: cp.asnumpy(a), n_repeat=10, n_warmup=3)
    ms = cpu_ms(r); gbps = (n*4) / (ms/1e3) / 1e9
    print(f'N={n:>12,} | size={n*4/1e6:6.2f} GB | {ms:8.3f} ms | ~{gbps:5.2f} GB/s')
# 관찰: 작은 N은 고정 오버헤드 지배(대역폭 낮게 측정), 큰 N에서 실제 PCIe 대역폭에 근접

<a id="9"></a>
## 9. 체크포인트

- [ ] `print_env()`에 GPU 이름과 VRAM이 출력됨
- [ ] CPU(지연시간)와 GPU(처리량)의 차이를 한 문장으로 설명할 수 있음
- [ ] GPU 연산이 **비동기**임을 이해하고, `bench`로 올바르게 측정함
- [ ] `asnumpy`(전송)가 비용이 크다는 것을 수치로 확인함
- [ ] 연습문제 `feature_gpu`가 `assert_allclose`를 통과함

다음 노트북: **`01_benchmark_basics`** — NumPy vs CuPy를 크기별로 벤치마크하고 GPU 가속의 손익분기점을 찾습니다.